# EDA: 2022 Inflation and Rate-Hike Bear Market

**Owner:** Fatima Nasir · **Week 1** · Market Stress Early-Warning System

**Goal:** understand what U.S. equity market stress looked like in 2022 at the stock level, check data quality and survivorship for this period, and test whether co-movement among S&P 500 stocks rose *before* the worst of the stress or only during it (Section 7). 2022 is our test of a **slow grind**: a long decline driven by inflation and interest rates, with lower peak volatility than 2008 or 2020. The team's stress rule splits it into three episodes (Feb to Mar, Apr to Jun, Sep to Nov), shaded on the charts.

**Before running:** this notebook reads `data/processed/` (daily, point-in-time S&P 500 member prices). Either run `python scripts/download_sp500_prices.py` from the repo root, or copy the team's shared `data/processed/` folder into the repo. If `data/processed/daily_panel.parquet` from `01_data_collection.ipynb` exists, VIX and the St. Louis Fed Financial Stress Index are added.

**Window:** 2022-01-01 to 2022-12-31, with 2021 loaded as warm-up for rolling windows and as the calm baseline.

**What changed from the first draft:** the first draft used Brandon's *weekly* `^GSPC` file, could not run top to bottom (the import cell was missing), and could not compute stock-level coverage, dispersion, breadth or connectedness because `data/processed/` was not on that computer. This version uses daily member prices and follows the 2008 and 2011 notebooks so the four periods can be compared. The first draft's findings are kept below Section 9.

**Reference dates.** The market peak and trough are computed from the data in Section 3, not typed in. The event dates below are from memory and **must be checked against a primary source before they go in the report**:

| Date | Event | To verify with |
|---|---|---|
| 2022-02-24 | Russia's invasion of Ukraine begins | News or government source |
| 2022-03-16 | First Federal Reserve rate increase of the cycle | Federal Reserve press release |
| 2022-06-15 | Federal Reserve raises rates by 0.75 percentage points | Federal Reserve press release |

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src import prices as px

PROC = ROOT / "data" / "processed"
if not (PROC / "sp500_close.parquet").exists():
    raise FileNotFoundError(
        f"{PROC / 'sp500_close.parquet'} not found. From the repo root run "
        "`python scripts/download_sp500_prices.py`, or copy the team's shared data/processed/ folder into the repo."
    )

YEAR = 2022
WARMUP, START, END = "2021-01-01", "2022-01-01", "2022-12-31"
CALM, CALM_LABEL = ("2021-01-01", "2021-12-31"), "2021"   # calm comparison period
ACUTE = ("2022-04-20", "2022-11-25")                       # used for the coverage check

# Verify each date against a primary source before citing it (see the table above)
EVENTS = {
    "2022-02-24": "Russia invades Ukraine",
    "2022-03-16": "First Fed hike",
    "2022-06-15": "Fed +0.75 pt",
}

# Sub-periods for the connectedness table (they follow the three stress episodes)
PERIODS = {
    "Jan to Feb 2022": ("2022-01-01", "2022-02-28"),
    "Mar to Jun 2022 (first leg)": ("2022-03-01", "2022-06-30"),
    "Jul to Aug 2022 (summer rally)": ("2022-07-01", "2022-08-31"),
    "Sep to Oct 2022 (second leg)": ("2022-09-01", "2022-10-31"),
    "Nov to Dec 2022": ("2022-11-01", "2022-12-31"),
}

# Baselines for the timing table: (baseline period, search window)
BASELINES = {
    "A: 2021": (slice("2021-01-01", "2021-12-31"), slice(START, END)),
    "B: Jul-Dec 2021": (slice("2021-07-01", "2021-12-31"), slice(START, END)),
}

# Stress episodes from the team's label rule (docs/stress_label_rule.md), shaded on the market charts
episodes = pd.read_csv(ROOT / "docs" / "stress_periods.csv", parse_dates=["start", "end"])
episodes = episodes[(episodes["end"] >= START) & (episodes["start"] <= END)]

def mark_events(ax, labels=True):
    for d, lab in EVENTS.items():
        ax.axvline(pd.Timestamp(d), color="grey", ls="--", lw=0.8)
        if labels:
            ax.text(pd.Timestamp(d), 1.0, f" {lab}", transform=ax.get_xaxis_transform(),
                    rotation=90, va="top", ha="left", fontsize=7, color="grey")

def shade_episodes(ax):
    for _, ep in episodes.iterrows():
        ax.axvspan(max(ep["start"], pd.Timestamp(START)), min(ep["end"], pd.Timestamp(END)),
                   color="tab:red", alpha=0.08, lw=0)

pd.set_option("display.width", 140)
print("Stress episodes in this window:")
display(episodes[["episode_id", "start", "start_trailing", "end", "name"]])

## 1. Load data

In [ ]:
close = pd.read_parquet(PROC / "sp500_close.parquet").loc[WARMUP:END]
membership = pd.read_parquet(PROC / "sp500_membership.parquet").loc[WARMUP:END]
report = pd.read_csv(PROC / "sp500_coverage_report.csv", index_col="ticker",
                     parse_dates=["first_price", "last_price", "first_member_day"])

# Daily simple returns from unmasked prices, then keep member days only.
stock_cols = membership.columns
rets_all = close.reindex(columns=stock_cols).pct_change(fill_method=None)
rets = rets_all.where(membership)

gspc = close["^GSPC"]
print(f"{close.index.min().date()} to {close.index.max().date()}: {len(close)} trading days")
print(f"{len(stock_cols)} tickers were members at some point in the full download period")

## 2. Data quality: how much of the index do we actually see?

Survivorship check: on each day, what share of the stocks that were *in the index that day* have a Yahoo price?

In [ ]:
cov = px.coverage_by_day(close, membership).loc[START:END]

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(cov.index, cov["share_priced"] * 100)
ax.set_ylabel("% of members with a price")
ax.set_title(f"Share of S&P 500 members with Yahoo price data, {YEAR}")
mark_events(ax)
plt.tight_layout(); plt.show()

acute = cov.loc[ACUTE[0]:ACUTE[1]]
print(f"Average share priced, full window: {cov['share_priced'].mean():.1%}")
print(f"Average share priced, {ACUTE[0]} to {ACUTE[1]}: {acute['share_priced'].mean():.1%}")
print(f"Lowest daily share: {cov['share_priced'].min():.1%} on {cov['share_priced'].idxmin().date()}")

In [ ]:
# Members during the window with no or partial price data, ranked by days in the index during the window
win_mem = membership.loc[START:END]
win_has = close.reindex(columns=stock_cols).loc[START:END].notna() & win_mem
gap = pd.DataFrame({
    "member_days_in_window": win_mem.sum(),
    "priced_days_in_window": win_has.sum(),
})
gap = gap[gap["member_days_in_window"] > 0]
gap["coverage_in_window"] = (gap["priced_days_in_window"] / gap["member_days_in_window"]).round(3)
missing = gap[gap["coverage_in_window"] < 1].sort_values(["member_days_in_window", "coverage_in_window"],
                                                         ascending=[False, True])
info_cols = [c for c in ["first_price", "last_price", "late_start", "check_ticker_reuse", "prices_from"] if c in report]
missing = missing.join(report[info_cols], how="left")
print(f"{len(missing)} of {len(gap)} window members have incomplete prices; "
      f"{int((gap['priced_days_in_window'] == 0).sum())} have none")
missing.head(40)

*For the write-up:* look at the names above. Are the missing ones firms that failed or were acquired after this period, or renamed tickers (the `prices_from` column shows renames we already fixed)? Well-known companies that still trade under a new symbol can be added to `TICKER_ALIASES` in `src/universe.py`.

### 2b. Price sanity: implausible daily returns

Member-days with an absolute daily return above 40% are listed and those tickers are excluded from the stock-level measures below. These are usually ticker reuse (a symbol that later belonged to another company). In 2022 some real stocks did move a lot, so look at each one before deciding it is an error.

In [ ]:
JUMP = 0.40
r_win = rets.loc[START:END]
jumps = r_win.stack()
jumps = jumps[jumps.abs() > JUMP].rename("daily_return").reset_index()
jumps.columns = ["date", "ticker", "daily_return"]
print(jumps.sort_values(["ticker", "date"]).to_string(index=False) if len(jumps) else "No member-day moves above 40%.")

BAD_TICKERS = sorted(jumps["ticker"].unique())
print(f"\nExcluded from stock-level measures: {BAD_TICKERS}")
rets = rets.drop(columns=BAD_TICKERS)
stock_cols = stock_cols.drop(BAD_TICKERS)

## 3. Market overview: level, drawdown and volatility

In [ ]:
g = gspc.loc[START:END]
drawdown = g / g.cummax() - 1
daily = gspc.pct_change(fill_method=None)
vol20 = daily.rolling(20).std().loc[START:END] * np.sqrt(252)

fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
axes[0].plot(g.index, g); axes[0].set_ylabel("^GSPC level")
axes[1].fill_between(drawdown.index, drawdown * 100, 0, color="tab:red", alpha=0.4)
axes[1].set_ylabel("Drawdown from peak (%)")
axes[2].plot(vol20.index, vol20 * 100, color="tab:orange"); axes[2].set_ylabel("20-day realized vol (ann., %)")
for i, ax in enumerate(axes):
    shade_episodes(ax)
    mark_events(ax, labels=(i == 0))
axes[0].set_title(f"S&P 500 index, {YEAR} (shaded = stress episodes from the team's label rule)")
plt.tight_layout(); plt.show()

trough_date = drawdown.idxmin()
peak_date = g.loc[:trough_date].idxmax()
print(f"Peak close before the drawdown: {g[peak_date]:,.2f} on {peak_date.date()}")
print(f"Trough close: {g[trough_date]:,.2f} on {trough_date.date()}")
print(f"Maximum drawdown: {drawdown.min():.1%} over {len(g.loc[peak_date:trough_date]) - 1} trading days")
print(f"Highest 20-day realized vol: {vol20.max():.1%} on {vol20.idxmax().date()}")
print(f"Average 20-day realized vol, {CALM_LABEL}: {(daily.rolling(20).std() * np.sqrt(252)).loc[CALM[0]:CALM[1]].mean():.1%}")

### 3b. Returns and recovery

Carried over from the first (weekly) draft of this notebook, now on daily closes.

In [ ]:
g_rets = daily.loc[START:END]
first, last = g.iloc[0], g.iloc[-1]
recovered = g.loc[trough_date:][g.loc[trough_date:] >= g[peak_date]]
vol_prev = daily.loc[CALM[0]:CALM[1]].std() * np.sqrt(252)
vol_year = g_rets.std() * np.sqrt(252)

print(f"First-to-last close in {YEAR}: {first:,.2f} -> {last:,.2f} ({last / first - 1:+.1%}, price index, no dividends)")
print(f"First close back at or above the pre-drawdown peak: "
      f"{recovered.index[0].date() if len(recovered) else 'not within this window'}")
print(f"Annualized daily volatility: {CALM_LABEL} {vol_prev:.1%}, {YEAR} {vol_year:.1%} ({vol_year / vol_prev:.2f}x)")
print(f"$10,000 at the peak close would have fallen to about ${10_000 * (1 + drawdown.min()):,.0f} at the trough "
      "(index level only, no fees or dividends)")
print("\nFive worst and five best daily returns of ^GSPC (%):")
ranked = (g_rets * 100).round(2).sort_values()
display(pd.concat([ranked.head(5), ranked.tail(5)]).to_frame("daily return (%)"))

### 3c. VIX and the St. Louis Fed Financial Stress Index (optional)

Runs only if `daily_panel.parquet` from `01_data_collection.ipynb` exists. Both series are FRED data. `STLFSI4` is weekly and is forward-filled onto trading days by observation date, which is fine for EDA but **not** for modeling (release lag, see `01_data_collection.ipynb`).

In [ ]:
panel_path = PROC / "daily_panel.parquet"
vix = stl = None
if panel_path.exists():
    panel = pd.read_parquet(panel_path)
    if "VIXCLS" in panel:
        vix = panel["VIXCLS"].reindex(close.index).ffill()
    if "STLFSI4" in panel:
        stl = panel["STLFSI4"].ffill().reindex(close.index, method="ffill")

if vix is None and stl is None:
    print("daily_panel.parquet not found; run 01_data_collection.ipynb to add VIX and STLFSI4.")
else:
    fig, ax1 = plt.subplots(figsize=(12, 4))
    if vix is not None:
        v = vix.loc[START:END]
        ax1.plot(v.index, v, color="tab:orange", label="VIX close"); ax1.set_ylabel("VIX")
        print(f"VIX peak close: {v.max():.1f} on {v.idxmax().date()}")
    if stl is not None:
        ax2 = ax1.twinx()
        s = stl.loc[START:END]
        ax2.plot(s.index, s, color="tab:red", label="STLFSI4"); ax2.set_ylabel("STLFSI4", color="tab:red")
        print(f"STLFSI4 peak: {s.max():.2f} on {s.idxmax().date()}")
    shade_episodes(ax1); mark_events(ax1)
    ax1.set_title(f"VIX and St. Louis Fed Financial Stress Index, {YEAR}")
    plt.tight_layout(); plt.show()

## 4. Sanity check: equal-weighted member portfolio vs the index

The official index is cap-weighted, so the two lines should move together but not match. A large, persistent gap would point to a data problem. `^GSPC` is a price index while member returns come from dividend-adjusted closes, so the equal-weighted line should end slightly higher even with identical stocks.

In [ ]:
ew = rets.loc[START:END].mean(axis=1, skipna=True)
ew_level = (1 + ew.fillna(0)).cumprod()
ix_level = g / g.iloc[0]

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(ix_level.index, ix_level, label="^GSPC (cap-weighted)")
ax.plot(ew_level.index, ew_level, label="Equal-weighted members with prices")
ax.set_ylabel("Growth of $1"); ax.legend(); mark_events(ax, labels=False)
ax.set_title(f"Equal-weighted member portfolio vs S&P 500, {YEAR}")
plt.tight_layout(); plt.show()

daily_corr = ew.corr(g_rets)
print(f"Correlation of daily returns (EW vs ^GSPC): {daily_corr:.3f}")
print(f"{YEAR} total return: EW {ew_level.iloc[-1] - 1:.1%}, ^GSPC price index {ix_level.iloc[-1] - 1:.1%}")

## 5. Cross-section: dispersion and breadth

* **Dispersion:** standard deviation of returns across stocks each day.
* **Breadth:** share of members trading above their own 50-day moving average.

In [ ]:
dispersion_full = rets.std(axis=1, skipna=True)
disp5_full = dispersion_full.rolling(5).mean()
disp5 = disp5_full.loc[START:END]
ma50 = close.reindex(columns=stock_cols).rolling(50, min_periods=50).mean()
above = (close.reindex(columns=stock_cols) > ma50).where(membership[stock_cols] & ma50.notna())
breadth_full = above.mean(axis=1, skipna=True)
breadth = breadth_full.loc[START:END]

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(disp5.index, disp5 * 100)
axes[0].set_ylabel("Cross-sectional std (%), 5d avg")
axes[1].plot(breadth.index, breadth * 100, color="tab:green")
axes[1].set_ylabel("% of members above 50d MA")
for i, ax in enumerate(axes):
    shade_episodes(ax)
    mark_events(ax, labels=(i == 0))
axes[0].set_title(f"Dispersion and breadth, {YEAR}")
plt.tight_layout(); plt.show()

print(f"Dispersion (5d avg): {CALM_LABEL} mean {disp5_full.loc[CALM[0]:CALM[1]].mean():.2%}, "
      f"peak {disp5.max():.2%} on {disp5.idxmax().date()}")
print(f"Breadth: {CALM_LABEL} mean {breadth_full.loc[CALM[0]:CALM[1]].mean():.1%}, "
      f"low {breadth.min():.1%} on {breadth.idxmin().date()}")

## 6. Connectedness: did stocks start moving together before the worst of the stress?

For every 5th trading day, take the previous 60 trading days of returns for stocks that were members with complete data over that window, and compute the **average pairwise correlation** and the **first principal component variance share** (largest eigenvalue / number of stocks). Both measure co-movement *among S&P 500 stocks*, not across asset classes.

In [ ]:
def connectedness(returns: pd.DataFrame, window: int = 60, step: int = 5, min_stocks: int = 100) -> pd.DataFrame:
    rows = {}
    idx = returns.index
    for end in range(window, len(idx) + 1, step):
        block = returns.iloc[end - window:end]
        block = block.loc[:, block.notna().all()]          # complete data over the window only
        block = block.loc[:, block.std() > 0]
        n = block.shape[1]
        if n < min_stocks:
            continue
        c = np.corrcoef(block.values, rowvar=False)
        avg = (c.sum() - n) / (n * (n - 1))
        top = np.linalg.eigvalsh(c)[-1] / n
        rows[idx[end - 1]] = {"avg_corr": avg, "pc1_share": top, "n_stocks": n}
    return pd.DataFrame(rows).T.astype(float)

conn_full = connectedness(rets.loc[WARMUP:END])
conn = conn_full.loc[START:END]

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
axes[0].plot(conn.index, conn["avg_corr"]); axes[0].set_ylabel("Avg pairwise corr (60d)")
axes[1].plot(conn.index, conn["pc1_share"], color="tab:purple"); axes[1].set_ylabel("PC1 variance share (60d)")
axes[2].plot(conn.index, conn["n_stocks"], color="grey"); axes[2].set_ylabel("Stocks used")
for i, ax in enumerate(axes):
    shade_episodes(ax)
    mark_events(ax, labels=(i == 0))
axes[0].set_title(f"Stock-level connectedness (S&P 500 members), {YEAR}")
plt.tight_layout(); plt.show()

Average correlation by sub-period. Each value is a trailing 60-day window ending on that date, so a window ending early in a sub-period mostly covers the one before.

In [ ]:
period_stats = pd.DataFrame({
    name: {"mean avg_corr": conn.loc[a:b, "avg_corr"].mean(),
           "min": conn.loc[a:b, "avg_corr"].min(),
           "max": conn.loc[a:b, "avg_corr"].max(),
           "mean PC1 share": conn.loc[a:b, "pc1_share"].mean()}
    for name, (a, b) in PERIODS.items()
}).T.round(3)
display(period_stats)

jump = conn["avg_corr"].diff()
print(f"Largest single-step rise: +{jump.max():.3f} in the window ending {jump.idxmax().date()}")
print(f"Peak: {conn['avg_corr'].max():.3f} on {conn['avg_corr'].idxmax().date()} "
      f"(the 20-day volatility peak was {vol20.idxmax().date()})")

## 7. Timing table: when did each measure first get unusual?

For each measure, the first date its value exceeded a baseline mean by 2 standard deviations, plus its peak date. This is a rough timing comparison, not a lead-time test (that comes in Week 6). The 2-standard-deviation rule is an arbitrary EDA choice, and the answer can change with the baseline (see the 2011 notebook), so compare against both stress-episode start dates printed below.

In [ ]:
vol20_full = daily.rolling(20).std() * np.sqrt(252)

measures = {
    "20d realized vol": vol20_full,
    "dispersion (5d avg)": disp5_full,
    "avg pairwise corr": conn_full["avg_corr"],
    "PC1 variance share": conn_full["pc1_share"],
}
if vix is not None:
    measures["VIX"] = vix
if stl is not None:
    measures["STLFSI4"] = stl

rows = []
for bname, (pre, search) in BASELINES.items():
    for name, s in measures.items():
        s = s.astype(float)
        base = s.loc[pre].dropna()
        thr = base.mean() + 2 * base.std()
        w = s.loc[search]
        hit = w[w > thr]
        rows.append({
            "baseline": bname, "measure": name,
            "baseline mean": round(base.mean(), 4),
            "threshold (mean + 2 sd)": round(thr, 4),
            "first date above threshold": hit.index[0].date() if len(hit) else None,
            f"peak date ({YEAR})": s.loc[START:END].idxmax().date(),
            "peak value": round(s.loc[START:END].max(), 4),
        })
timing = pd.DataFrame(rows).set_index(["baseline", "measure"])
print("Stress episode start dates in this window (centered / trailing):",
      [(str(a.date()), str(b)) for a, b in zip(episodes["start"], episodes["start_trailing"])])
timing

## 8. Robustness: does a fixed, survivor-style panel change the connectedness picture?

Section 6 uses a rolling universe. Here we recompute it using only stocks that were members **and** had a price on **every** trading day of 2022. If the two lines differ a lot, the network results are sensitive to how we handle index changes and missing firms.

In [ ]:
n_days = len(win_mem)
fixed = gap.index[(gap["member_days_in_window"] == n_days) & (gap["priced_days_in_window"] == n_days)]
fixed = fixed.difference(BAD_TICKERS)
print(f"Fixed panel: {len(fixed)} stocks present and priced on all {n_days} trading days of {YEAR}")

conn_fixed = connectedness(rets.loc[WARMUP:END, fixed]).loc[START:END]

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(conn.index, conn["avg_corr"], label="All members with data in each window")
ax.plot(conn_fixed.index, conn_fixed["avg_corr"], label=f"Fixed panel (present all of {YEAR})", ls="--")
ax.set_ylabel("Avg pairwise corr (60d)"); ax.legend(); mark_events(ax, labels=False)
plt.tight_layout(); plt.show()

both = conn[["avg_corr"]].join(conn_fixed[["avg_corr"]], rsuffix="_fixed").dropna()
diff = (both["avg_corr"] - both["avg_corr_fixed"]).abs()
print(f"Max absolute difference in avg corr: {diff.max():.4f} (window ending {diff.idxmax().date()})")

## 9. Observations (fill in after running)

Write what the outputs show, not what we expect. Cite the section each number comes from, as in the 2008 and 2011 notebooks. One or two sentences each.

1. **Data coverage:** what share of members had prices, and are the missing names failed, acquired or renamed firms? (Section 2)
2. **Drawdown and volatility:** peak, trough, size and speed of the decline; volatility peak and date. (Sections 3 and 3b)
3. **Connectedness:** did average correlation or the PC1 share rise before the worst of the stress, or mostly after? (Sections 6 and 7)
4. **Timing:** compare first-crossing dates in Section 7 with the stress episode start dates (centered and trailing).
5. **Cross-section:** what happened to dispersion and breadth around the key dates? (Section 5)
6. **Robustness and sanity checks:** EW vs index correlation and the fixed-panel difference. (Sections 4 and 8)
7. **Implications for the model:** what this period suggests about window length, features, or how we label it.

### First draft findings (weekly data, Fatima, October 2026)

These came from weekly closes labeled by week **start** date, so dates are approximate and daily extremes are larger. Compare them with Sections 3 and 3b above, then keep or drop them.

* First-to-last weekly close in 2022: 4,677.03 to 3,839.50, **-17.91%** (price change, not a total return).
* Maximum drawdown from the running weekly peak: **-23.39%**.
* Worst weekly return: **-5.79%**.
* Highest 12-week rolling annualized volatility of weekly returns: **30.33%**.

## Save outputs for the team

In [ ]:
out = PROC / f"eda_{YEAR}"
out.mkdir(parents=True, exist_ok=True)
conn.to_csv(out / "connectedness_60d.csv")
conn_fixed.to_csv(out / "connectedness_60d_fixed_panel.csv")
cov.to_csv(out / "coverage_by_day.csv")
missing.to_csv(out / "members_missing_prices.csv")
jumps.to_csv(out / "implausible_returns.csv", index=False)
period_stats.to_csv(out / "avg_corr_by_period.csv")
timing.to_csv(out / "timing_table.csv")
print("Saved to", out.relative_to(ROOT))
for f in sorted(out.iterdir()):
    print("  ", f.name)